<a href="https://colab.research.google.com/github/rajivagarwal221/Aahar-Kitchen-System/blob/main/notebooks/train_aahar_packaged.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AHAAR — Packaged-food brand detector (v1)

A **separate** YOLOv11 model for branded packaged products found in Indian homes (Amul, Maggi, Parle-G, Aashirvaad, Tata Salt, Haldiram's …). It runs next to the ingredient detector; each brand class maps to a pantry ingredient where one exists (e.g. `aashirvaad_atta → atta_flour`), so recipe matching keeps working.

Class list: `datasets/packaged/classes.csv`. Plan and labelling guide: `datasets/packaged/README.md`.

Data comes from three places, merged against the class list:
1. **Roboflow Universe** datasets with real bounding boxes (CC BY 4.0).
2. **Your own photos** from Google Drive, one folder per class, one product per photo. Boxes are drawn automatically by YOLO-World and you review them on a contact sheet.
3. *(Optional)* **Open Food Facts** India front-of-pack photos, boxed the same way.

**Before Run all**
1. `Runtime → Change runtime type → T4 GPU`
2. Colab secrets: `ROBOFLOW_API_KEY` (free account), `HF_TOKEN` *(only for the upload cell)*
3. Put your photos at `DRIVE_PHOTOS_DIR` (cell 2), e.g. `MyDrive/AAhar/packaged_photos/maggi_noodles/IMG_001.jpg`.

Cells 1–9 take ~15 min and gate the long training in cell 10.

## 1. Install

In [ ]:
!pip install -q "ultralytics==8.3.*" roboflow "huggingface_hub==0.26.*" pyyaml requests 2>&1 | tail -3
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime → Change runtime type → T4 GPU, then re-run.'
print('GPU:', torch.cuda.get_device_name(0),
      round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')

## 2. Config

In [ ]:
from pathlib import Path

REPO_URL    = 'https://github.com/rajivagarwal221/Aahar-Kitchen-System.git'
REPO_BRANCH = 'main'          # use 'claude/project-thread-1r16i4' until the PR is merged
REPO_DIR    = Path('/content/aahar')

# Your own photos: <DRIVE_PHOTOS_DIR>/<class>/*.jpg  (class names from classes.csv)
DRIVE_PHOTOS_DIR = '/content/drive/MyDrive/AAhar/packaged_photos'
# Training runs land here (survives Colab disconnects)
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/AAhar/runs'
RUN_NAME         = 'aahar_packaged_v1'

# Roboflow Universe sources. version=None → newest version.
# 'ambiguous': source classes too vague to map (e.g. a bare 'Britannia' box could be
# Good Day or Marie Gold). Images containing one are dropped entirely, so the model
# never learns that a real product is background. Unmapped non-food classes
# (soap, shampoo …) are simply removed as boxes.
ROBOFLOW_SOURCES = [
    dict(workspace='hariom-jyhyf',               project='object-detection-8iyni',              version=None,
         ambiguous=['Britannia','Parle','Nestle','Tata','Cadbury','Kwality','Kissan','Dabur',
                    'Kellogs','TomatoKetchup','SoftDrink','Sunfeast','Amul']),
    dict(workspace='st-xaviers-college-zadwr',   project='supermarket-products-detection-yxgj6', version=None,
         ambiguous=[]),
    dict(workspace='agentsk47',                  project='indian-grocery-object-detection-mfsnx', version=None,
         ambiguous=[]),
    dict(workspace='nmims-b6ta9',                project='retail-store-india-new-2',             version=1,
         ambiguous=[]),
]

USE_OPEN_FOOD_FACTS = True    # front-of-pack photos from openfoodfacts.org (CC BY-SA)
OFF_PER_CLASS       = 40

# Auto-boxing (own photos + Open Food Facts) with open-vocabulary YOLO-World
AUTOBOX_MODEL   = 'yolov8s-worldv2.pt'
AUTOBOX_PROMPTS = ['food packet', 'bottle', 'box', 'jar', 'pouch', 'carton',
                   'tetra pack', 'can', 'tin', 'biscuit packet', 'bag']
AUTOBOX_CONF    = 0.05
AUTOBOX_MIN_AREA = 0.02       # ignore boxes under 2% of the image

MODEL      = 'yolo11m.pt'     # same family as the ingredient model
IMG_SIZE   = 640
EPOCHS     = 80
BATCH      = 16
VAL_OWN    = 0.25             # share of YOUR photos held out for val (real-home domain)
VAL_OTHER  = 0.08             # share of Roboflow / OFF images held out
MIN_TRAIN_PER_CLASS = 30      # classes below this are reported as weak

UPLOAD_TO_HF  = False         # leave False until you've checked the metrics
HF_MODEL_REPO = 'Rajiv221/aahar-packaged-detector'

IMG_EXT = {'.jpg', '.jpeg', '.png', '.webp'}
print('Config OK.')

## 3. Clone repo and load the class list

In [ ]:
import os, re, csv, json, subprocess

if not REPO_DIR.exists():
    subprocess.run(['git','clone','--branch',REPO_BRANCH,'--depth','1',REPO_URL,str(REPO_DIR)], check=True)

def norm(s):
    """'Parle-G', 'parle_g', 'Parle G' → 'parleg'"""
    return re.sub(r'[^a-z0-9]', '', str(s).lower())

CLASSES = list(csv.DictReader(open(REPO_DIR/'datasets/packaged/classes.csv', encoding='utf-8')))
class_names = [r['class'] for r in CLASSES]
cls_id = {n: i for i, n in enumerate(class_names)}
ALIAS = {}
for r in CLASSES:
    for a in [r['class']] + [x for x in r['aliases'].split('|') if x.strip()]:
        k = norm(a)
        if k in ALIAS and ALIAS[k] != r['class']:
            raise ValueError(f'Alias {a!r} maps to both {ALIAS[k]} and {r["class"]}')
        ALIAS[k] = r['class']
PANTRY = {r['class']: r['pantry_ingredient'] for r in CLASSES if r['pantry_ingredient']}
print(f'{len(class_names)} classes, {len(ALIAS)} aliases, {len(PANTRY)} map to a pantry ingredient')

# Every image that will be trained on: dict(img=Path, labels=[(cls, xc, yc, w, h)], source=str, own=bool)
SAMPLES = []

## 4. Download Roboflow Universe datasets

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

RF_ROOT = Path('/content/raw/roboflow'); RF_ROOT.mkdir(parents=True, exist_ok=True)
rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
rf_dirs = []
for src in ROBOFLOW_SOURCES:
    dst = RF_ROOT / src['project']
    try:
        if not (dst/'data.yaml').exists():
            proj = rf.workspace(src['workspace']).project(src['project'])
            ver = src['version']
            if ver is None:
                ver = max(int(str(v.version).rstrip('/').split('/')[-1]) for v in proj.versions())
            try:
                proj.version(ver).download('yolov11', location=str(dst), overwrite=True)
            except Exception:
                proj.version(ver).download('yolov8', location=str(dst), overwrite=True)
            print(f'OK   {src["project"]} v{ver}')
        else:
            print(f'SKIP {src["project"]} (already downloaded)')
        rf_dirs.append((src, dst))
    except Exception as e:
        print(f'FAIL {src["project"]}: {e}  (continuing without it)')

## 5. Remap Roboflow labels onto our classes

In [ ]:
import yaml
from collections import Counter

def load_names(data_yaml):
    names = yaml.safe_load(open(data_yaml))['names']
    return names if isinstance(names, list) else [names[k] for k in sorted(names, key=int)]

unmapped = Counter()
for src, root in rf_dirs:
    names = load_names(root/'data.yaml')
    amb = {norm(a) for a in src['ambiguous']}
    kept = dropped_amb = 0
    for img in root.rglob('*'):
        if img.suffix.lower() not in IMG_EXT or img.parent.name != 'images':
            continue
        lab = img.parent.parent / 'labels' / (img.stem + '.txt')
        if not lab.exists():
            continue
        out, bad = [], False
        for line in lab.read_text().split('\n'):
            p = line.split()
            if len(p) < 5:
                continue
            raw = names[int(p[0])]
            if norm(raw) in amb:
                bad = True; break
            tgt = ALIAS.get(norm(raw))
            if tgt is None:
                unmapped[f'{src["project"]}:{raw}'] += 1
                continue
            if len(p) > 5:    # polygon → bbox
                xs, ys = list(map(float, p[1::2])), list(map(float, p[2::2]))
                x1, x2, y1, y2 = min(xs), max(xs), min(ys), max(ys)
                box = ((x1+x2)/2, (y1+y2)/2, x2-x1, y2-y1)
            else:
                box = tuple(map(float, p[1:5]))
            out.append((tgt, *box))
        if bad:
            dropped_amb += 1; continue
        if out:
            SAMPLES.append(dict(img=img, labels=out, source=src['project'], own=False)); kept += 1
    print(f'{src["project"]:42s} kept {kept:5d} images, dropped {dropped_amb} with ambiguous classes')

print('\nTop unmapped source classes (add to aliases in classes.csv if they belong):')
for k, v in unmapped.most_common(40):
    print(f'  {v:5d}  {k}')

## 6. Auto-boxer (YOLO-World)

Draws one box around the main product in a photo. Your photos should show **one product per photo** (several angles, shelves, fridge door, lighting) so the biggest confident box is the right one.

In [ ]:
from ultralytics import YOLOWorld
from PIL import Image, ImageOps

boxer = YOLOWorld(AUTOBOX_MODEL)
boxer.set_classes(AUTOBOX_PROMPTS)

def autobox(path):
    """Return ((xc, yc, w, h) normalised, how) for the main product in the photo."""
    r = boxer.predict(str(path), conf=AUTOBOX_CONF, verbose=False)[0]
    if r.boxes is not None and len(r.boxes):
        xywhn = r.boxes.xywhn.cpu().numpy()
        conf = r.boxes.conf.cpu().numpy()
        area = xywhn[:, 2] * xywhn[:, 3]
        ok = area >= AUTOBOX_MIN_AREA
        if ok.any():
            score = conf * area ** 0.5 * ok    # favour confident AND large
            i = int(score.argmax())
            return tuple(float(v) for v in xywhn[i]), 'yolo-world'
    return (0.5, 0.5, 0.96, 0.96), 'fallback'

def fix_orientation(src, dst):
    """Phone photos carry EXIF rotation; bake it in so boxes match pixels."""
    tmp = Path(str(dst) + '.tmp.jpg')
    with Image.open(src) as im:
        ImageOps.exif_transpose(im).convert('RGB').save(tmp, quality=92)
    tmp.replace(dst)

## 7. Your own photos from Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

OWN_FIXED = Path('/content/raw/own'); OWN_FIXED.mkdir(parents=True, exist_ok=True)
review = []   # rows for the contact sheet
photos = Path(DRIVE_PHOTOS_DIR)
if not photos.exists():
    print(f'No folder at {photos} — skipping own photos. (Strongly recommended: see README.)')
else:
    for d in sorted(p for p in photos.iterdir() if p.is_dir()):
        tgt = ALIAS.get(norm(d.name))
        if tgt is None:
            print(f'  ? folder {d.name!r} matches no class — skipped'); continue
        n = 0
        for img in sorted(d.rglob('*')):
            if img.suffix.lower() not in IMG_EXT and img.suffix.lower() not in {'.heic'}:
                continue
            if img.suffix.lower() == '.heic':
                print(f'  HEIC not supported, convert to JPG: {img.name}'); continue
            fixed = OWN_FIXED / f'{tgt}__{d.name}__{img.stem}.jpg'
            if not fixed.exists():
                try: fix_orientation(img, fixed)
                except Exception as e:
                    print(f'  unreadable {img}: {e}'); continue
            box, how = autobox(fixed)
            SAMPLES.append(dict(img=fixed, labels=[(tgt, *box)], source='own', own=True))
            review.append((fixed, tgt, box, how)); n += 1
        print(f'  {tgt:28s} {n:4d} photos')
    fb = sum(1 for r in review if r[3] == 'fallback')
    print(f'Own photos: {len(review)}  (auto-box fell back to whole image on {fb})')

## 8. *(Optional)* Open Food Facts — India front-of-pack photos

Noisy but free. Each image is a single product, boxed by the same auto-boxer. Off when `USE_OPEN_FOOD_FACTS=False`.

In [ ]:
import requests, time, hashlib

OFF_ROOT = Path('/content/raw/off'); OFF_ROOT.mkdir(parents=True, exist_ok=True)
HEADERS = {'User-Agent': 'aahar-packaged-detector/0.1 (research notebook)'}

def off_search(query, n):
    url = 'https://world.openfoodfacts.org/cgi/search.pl'
    params = dict(search_terms=query, countries_tags_en='india', json=1,
                  page_size=min(100, n*2), fields='code,product_name,brands,image_front_url')
    try:
        r = requests.get(url, params=params, headers=HEADERS, timeout=30)
        r.raise_for_status()
        return [p for p in r.json().get('products', []) if p.get('image_front_url')]
    except Exception as e:
        print(f'  OFF search failed for {query!r}: {e}'); return []

if USE_OPEN_FOOD_FACTS:
    for row in CLASSES:
        c = row['class']
        brand_k = norm(row['brand'].split()[0])
        q = f"{row['brand']} {row['product'].split('(')[0]}"
        prods = [p for p in off_search(q, OFF_PER_CLASS) if brand_k in norm(p.get('brands', ''))]
        got = 0
        for p in prods[:OFF_PER_CLASS]:
            dst = OFF_ROOT / c / f"{p['code'] or hashlib.md5(p['image_front_url'].encode()).hexdigest()}.jpg"
            dst.parent.mkdir(exist_ok=True)
            if not dst.exists():
                try:
                    b = requests.get(p['image_front_url'], headers=HEADERS, timeout=30).content
                    dst.write_bytes(b); fix_orientation(dst, dst)
                except Exception:
                    dst.unlink(missing_ok=True); continue
            box, how = autobox(dst)
            SAMPLES.append(dict(img=dst, labels=[(c, *box)], source='openfoodfacts', own=False))
            review.append((dst, c, box, how)); got += 1
        print(f'  {c:28s} {got:3d} OFF images')
        time.sleep(1.0)   # be polite to the OFF API

## 9. Contact sheet — check the auto-drawn boxes

Look through the grid. Wrong-product or badly boxed photos: delete them from Drive (or Open Food Facts cache) and re-run cells 3, 5, 7, 8, 9.

In [ ]:
import random
from PIL import ImageDraw

def contact_sheet(rows, n=60, cols=10, cell=192, out='/content/review.jpg'):
    rows = random.Random(0).sample(rows, min(n, len(rows)))
    if not rows: print('Nothing to review.'); return
    sheet = Image.new('RGB', (cols*cell, ((len(rows)+cols-1)//cols)*(cell+16)), 'white')
    for k, (p, c, (xc, yc, w, h), how) in enumerate(rows):
        im = Image.open(p).convert('RGB'); W, H = im.size
        d = ImageDraw.Draw(im)
        d.rectangle([(xc-w/2)*W, (yc-h/2)*H, (xc+w/2)*W, (yc+h/2)*H],
                    outline='red' if how == 'fallback' else 'lime', width=max(3, W//100))
        im.thumbnail((cell, cell))
        x, y = (k % cols)*cell, (k//cols)*(cell+16)
        sheet.paste(im, (x, y)); ImageDraw.Draw(sheet).text((x+2, y+cell), c[:30], fill='black')
    sheet.save(out); print('Saved', out)
    from IPython.display import display; display(sheet)

contact_sheet(review)

## 10. Build the YOLO dataset and sanity-check

Your own photos are weighted toward **val** so the score reflects real kitchens, not shop shelves.

In [ ]:
import shutil

DATA = Path('/content/packaged_data')
if DATA.exists(): shutil.rmtree(DATA)
for sub in ('images/train', 'images/val', 'labels/train', 'labels/val'):
    (DATA/sub).mkdir(parents=True)

rng = random.Random(42)
counts = {'train': Counter(), 'val': Counter()}
for k, s in enumerate(SAMPLES):
    split = 'val' if rng.random() < (VAL_OWN if s['own'] else VAL_OTHER) else 'train'
    stem = f"{s['source'][:12]}_{k:06d}"
    shutil.copy2(s['img'], DATA/f'images/{split}'/f"{stem}{s['img'].suffix.lower()}")
    lines = []
    for c, xc, yc, w, h in s['labels']:
        xc, yc = min(max(xc, 0), 1), min(max(yc, 0), 1)
        w, h = min(max(w, 1e-3), 1), min(max(h, 1e-3), 1)
        lines.append(f'{cls_id[c]} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}')
        counts[split][c] += 1
    (DATA/f'labels/{split}'/f'{stem}.txt').write_text('\n'.join(lines) + '\n')

cfg_path = DATA/'packaged.yaml'
cfg_path.write_text(yaml.safe_dump({
    'path': str(DATA), 'train': 'images/train', 'val': 'images/val',
    'nc': len(class_names), 'names': class_names,
}, sort_keys=False))

n_tr = len(list((DATA/'images/train').iterdir())); n_va = len(list((DATA/'images/val').iterdir()))
print(f'images  train {n_tr}  val {n_va}   boxes train {sum(counts["train"].values())}  val {sum(counts["val"].values())}')
print(f'\n{"class":28s} {"train":>6s} {"val":>5s}')
weak, empty = [], []
for c in class_names:
    tr, va = counts['train'][c], counts['val'][c]
    flag = '  <- no data' if tr == 0 else ('  <- weak' if tr < MIN_TRAIN_PER_CLASS else '')
    print(f'{c:28s} {tr:6d} {va:5d}{flag}')
    (empty if tr == 0 else weak if tr < MIN_TRAIN_PER_CLASS else []).append(c)
print(f'\n{len(class_names)-len(empty)-len(weak)} classes OK, {len(weak)} weak, {len(empty)} with no data.')
print('Photograph the weak / empty ones at home (see README) before trusting the model on them.')
if n_tr < 200 or n_va < 20:
    print('\n❌ ABORT: too little data to train. Do NOT run cell 11.')
else:
    print('\n✅ OK to train.')

## 11. Train (~1.5–2.5 h on a T4) — re-run to continue after a disconnect

In [ ]:
from ultralytics import YOLO

out = Path(DRIVE_OUTPUT_DIR); out.mkdir(parents=True, exist_ok=True)
last_pt = out / RUN_NAME / 'weights' / 'last.pt'
if last_pt.exists():
    print('Resuming from', last_pt)
    try:
        YOLO(str(last_pt)).train(resume=True)
    except Exception as e:      # e.g. the run already finished
        print('Not resumed:', e)
else:
    model = YOLO(MODEL)
    results = model.train(
        data=str(cfg_path), epochs=EPOCHS, imgsz=IMG_SIZE, batch=BATCH, device=0,
        project=str(out), name=RUN_NAME, exist_ok=True,
        patience=20, cache='disk', amp=True, save_period=5,
        # Brand text and logos are not mirror-symmetric: no horizontal flip.
        fliplr=0.0, degrees=10.0, hsv_v=0.5, scale=0.6, mosaic=1.0, close_mosaic=10,
    )
best = out / RUN_NAME / 'weights' / 'best.pt'
print('Best:', best, 'exists:', best.exists())

## 12. Evaluate — per-class precision / recall

In [ ]:
import pandas as pd
m = YOLO(str(best)).val(data=str(cfg_path), imgsz=IMG_SIZE, split='val', verbose=False)
print(f'mAP50 {m.box.map50:.3f}   mAP50-95 {m.box.map:.3f}')
rows = []
for i, c in enumerate(m.ap_class_index):
    p, r, ap50, ap = m.box.class_result(i)
    rows.append(dict(cls=class_names[c], P=round(p,3), R=round(r,3), mAP50=round(ap50,3),
                     val_boxes=counts['val'][class_names[c]], pantry=PANTRY.get(class_names[c], '')))
per_class = pd.DataFrame(rows).sort_values('mAP50')
per_class.to_csv(out/RUN_NAME/'per_class.csv', index=False)
print(per_class.to_string(index=False))

## 13. Try it on one photo

In [ ]:
from google.colab import files
up = files.upload()
for name in up:
    r = YOLO(str(best)).predict(name, conf=0.35, verbose=False)[0]
    for b in r.boxes:
        c = class_names[int(b.cls)]
        print(f'{c:28s} {float(b.conf):.2f}   pantry → {PANTRY.get(c, "(none)")}')
    from IPython.display import display; display(Image.fromarray(r.plot()[:, :, ::-1]))

## 14. Export ONNX and (optionally) upload to Hugging Face

Leaves `UPLOAD_TO_HF=False` by default. The backend does not load this model yet; wiring it in is a separate change.

In [ ]:
onnx_path = YOLO(str(best)).export(format='onnx', imgsz=IMG_SIZE, opset=12, simplify=True)
mapping = out/RUN_NAME/'pantry_map.json'
mapping.write_text(json.dumps({r['class']: {'brand': r['brand'], 'product': r['product'],
                               'category': r['category'], 'pantry': r['pantry_ingredient'] or None}
                               for r in CLASSES}, indent=1))
print('ONNX:', onnx_path, '\nmap :', mapping)

if UPLOAD_TO_HF:
    from huggingface_hub import HfApi, create_repo
    token = userdata.get('HF_TOKEN'); assert token, 'HF_TOKEN not set'
    create_repo(HF_MODEL_REPO, token=token, exist_ok=True, repo_type='model')
    api = HfApi(token=token)
    for p in [best, Path(onnx_path), cfg_path, mapping, out/RUN_NAME/'per_class.csv']:
        api.upload_file(path_or_fileobj=str(p), path_in_repo=p.name, repo_id=HF_MODEL_REPO)
        print('Uploaded', p.name)
    print(f'https://huggingface.co/{HF_MODEL_REPO}')
else:
    print('UPLOAD_TO_HF is False — nothing uploaded.')